<a href="https://colab.research.google.com/github/devwoo41/PromptEngineeringLecture/blob/master/13/RAG_exp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U \
langchain langchain-core langchain-huggingface langchain-community \
langchain-chroma langchain-text-splitters \
huggingface_hub transformers accelerate \
chromadb sentence-transformers bs4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 112.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 120.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 131.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.3/235.3 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from huggingface_hub import notebook_login
import os
notebook_login()


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. 모델 이름 설정
model_name = "meta-llama/Llama-3.2-1B-Instruct"

# 2. 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3. 모델 로드
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",          # GPU 있으면 자동으로 cuda 사용
    torch_dtype=torch.float16   # Colab GPU에서는 float16 권장
)

# 4. 추론 모드로 전환
#model.eval()

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [8]:
import torch
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

# transformers pipeline 생성
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    temperature=0.9,
    top_p=0.1,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
    return_full_text=False
)

# LangChain용 LLM 래핑
llm = HuggingFacePipeline(pipeline=pipe)

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'pad_token_id', 'max_new_tokens', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [9]:
print(llm.invoke("What is the FLEX?"))


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 FLEX is a type of flexible joint that allows for a wide range of motion, making it ideal for various applications, including sports, dance, and even everyday activities. FLEX joints are made from a combination of materials, such as polyurethane, polyethylene, and polypropylene, which provide flexibility and durability.

FLEX joints are designed to be lightweight, yet strong and resistant to wear and tear. They are often used in applications where a high degree of flexibility is required, such as in sports equipment, dance gear, and other products that need to move freely.

Some of the key benefits of FLEX joints include:

*


#### 번외 - Kybalion-1B-DPO 로 돌려보기

In [11]:
!pip install -U transformers accelerate langchain-huggingface huggingface_hub

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

# 1. 모델 이름
model_name = "devwoo/Kybalion-1B-DPO"

# 2. tokenizer 로드
tokenizer = AutoTokenizer.from_pretrained(model_name)

# pad_token 없으면 eos_token으로 설정
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3. model 로드
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

model.eval()

# 4. pipeline 생성
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
    return_full_text=False
)

# 5. LangChain LLM 래핑
llm = HuggingFacePipeline(pipeline=pipe)

response = llm.invoke("What is the FLEX?")
print(response)

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/180 [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 It stands for Flexible, Extensible, and Knowledgeable. It is a framework that helps organizations to be more innovative and efficient. It is a set of principles and guidelines that can be used to create a culture of innovation and knowledge sharing.
The FLEX framework is based on the following principles:
- Focus on the needs of customers and end-users
- Emphasize collaboration and communication
- Recognize the importance of data and analytics
- Encourage creativity and innovation
- Provide opportunities for knowledge sharing and exchange
- Foster a culture of continuous improvement and adaptation
The FLEX framework can be used to create a culture of innovation and knowledge sharing in any organization. It can be used to create a culture of continuous improvement and adaptation in any organization.
The FLEX framework is based on the following principles:
- Focus on the needs of customers and end-users
- Emphasize collaboration and communication
- Recognize the importance of data and a